In [1]:
import os
import glob

chain_index_path = os.path.join("ProAffinity-GNN", "data", "chain_index.txt")
fasta_dirs = [
    os.path.join("ProAffinity-GNN", "data", "FASTA", "mixed"),
    os.path.join("ProAffinity-GNN", "data", "FASTA", "2mole"),
    os.path.join("ProAffinity-GNN", "data", "FASTA", "3mole")
]

pdb_chains = {}
with open(chain_index_path, 'r') as f:
    for line in f:
        parts = line.strip().split('\t')
        if len(parts) == 2:
            pdb = parts[0].upper()
            chains = [c.strip() for c in parts[1].split(';') if c.strip()]
            pdb_chains[pdb] = chains

fixed_count = 0
for directory in fasta_dirs:
    if not os.path.exists(directory): continue
    
    fasta_files = glob.glob(os.path.join(directory, "*.fasta"))
    for file in fasta_files:
        filename = os.path.basename(file)
        pdb_id = filename.split('_')[0].upper() 
        
        try:
            file_idx = int(filename.split('_')[1].split('.')[0]) - 1
        except:
            file_idx = 0
        
        chain_id = "A" 
        if pdb_id in pdb_chains and file_idx < len(pdb_chains[pdb_id]):
            chain_id = pdb_chains[pdb_id][file_idx]

        with open(file, 'r') as f:
            lines = f.readlines()
        if not lines: continue
        
        sequence = "".join([l.strip() for l in lines[1:]])
        
        perfect_header = f">{pdb_id}_{file_idx+1}|Chain {chain_id}|Fake Protein|Fake Species"
        
        with open(file, 'w') as f:
            f.write(perfect_header + "\n" + sequence + "\n")
        fixed_count += 1

print(f" {fixed_count} fastas fixed")

 5088 fastas fixed
